# Lekcija 05 - Agentni RAG


## Postavljanje

Ovaj bilježnik prikazuje Agentic RAG (Retrieval-Augmented Generation) obrazac koristeći Microsoft Agent Framework.

**Preduvjeti:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — vaša krajnja točka Azure AI Search usluge
- `AZURE_SEARCH_API_KEY` — vaš Azure AI Search API ključ
- Azure OpenAI implementacija konfigurirana putem varijabli okruženja
- Azure CLI prijavljen (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Što je Agentic RAG?

Tradicionalni RAG slijedi fiksni tok: dohvatiti dokumente, zatim generirati odgovor. **Agentic RAG** ide korak dalje dajući agentu autonomiju da odluči **kada** i **kako** dohvatiti informacije.

S Agentic RAG-om, agent može:
- **Odlučiti** treba li dohvat podataka prije odgovaranja na pitanje
- **Izabrati** koji izvor podataka ili alat upitati
- **Procijeniti** dohvaćene rezultate i obaviti dodatni dohvat ako prvi pokušaj nije dovoljan
- **Povezati** informacije iz više koraka dohvata u koherentan odgovor

To čini agenta fleksibilnijim i preciznijim u usporedbi sa statičnim tokovima dohvaćanja i generiranja.


## Izrada alata za pretraživanje

U Agentic RAG-u, vanjski izvori podataka su umotani kao **alati** koje agent može pozvati prema potrebi. To agentu omogućuje da tretira dohvaćanje podataka kao još jednu radnju koju može poduzeti, umjesto kao obavezni korak.

Ispod definiramo bazu znanja o putovanjima i izlažemo je kao alat koji agent može pozvati za pronalaženje informacija o odredištu.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Izgradnja RAG agenta

Sada stvaramo agenta koji je uputno da **uvijek dohvaća informacije prije odgovaranja**. Agent koristi alat `search_travel_knowledge` kako bi utemeljio svoje odgovore u bazi znanja, umjesto da se oslanja na vlastite podatke za obuku.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iterativno dohvaćanje — obrazac Maker-Checker

Ključna prednost Agentic RAG-a je **iterativno dohvaćanje**. Agent može obaviti više rundi pretraživanja kako bi provjerio, usavršio ili proširio svoja početna otkrića — slično kao radni tok "maker-checker":

1. **Maker korak**: Agent dohvaća početne informacije i sastavlja odgovor.
2. **Checker korak**: Agent obavlja dodatna dohvaćanja kako bi provjerio detalje ili popunio praznine.

Ispod je agentu postavljeno pitanje koje zahtijeva usporedbu više odredišta, što ga potiče na višestruko pretraživanje.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Sažetak

U ovoj lekciji naučili ste kako izgraditi **Agentic RAG** sustav koristeći Microsoft Agent Framework:

- **Agentic RAG** omogućuje agentima da autonomno odlučuju kada će dohvatiti informacije, čineći dohvat dinamičnim umjesto fiksnim.
- **Alati kao izvori podataka**: Vanjske baze znanja (poput Azure AI Search) su obuhvaćene kao alati koje agent može pozvati.
- **Iterativni dohvat**: uzorak maker-checker omogućuje agentu izvođenje više krugova dohvata — pretraživanje, provjeru i usavršavanje — prije nego proizvede konačni odgovor.

U produkciji biste zamijenili memorijski `TRAVEL_KNOWLEDGE_BASE` stvarnim Azure AI Search indeksom za rukovanje velikim opsegom dokumenata o putovanjima.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Izjava o odricanju odgovornosti**:
Ovaj je dokument preveden korištenjem AI usluge prevođenja [Co-op Translator](https://github.com/Azure/co-op-translator). Iako nastojimo postići točnost, imajte na umu da automatski prijevodi mogu sadržavati pogreške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati službenim i autoritativnim izvorom. Za kritične informacije preporučuje se profesionalni ljudski prijevod. Ne snosimo odgovornost za bilo kakve nesporazume ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
